# Guiding Diagnostics Dashboard

***

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1". 

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.


## Introduction

The Guiding Diagnostics Dashboard provides insight into the guiding performance of one or more Roman WFI visits. It uses MAST visit and exposure metadata, engineering telemetry, and—when requested—guide window ASDF products. It plots noise-equivalent angles (NEAs) from fine-attitude correction estimate (FACE) telemetry, shows jitter from FACE telemetry or detector centroid telemetry, and displays guide window images with centroid positions overlaid. You can also export guide window and FACE ASDF products or create a guide window movie.

The dashboard does not use science ASDF products to make its plots.

## Requirements

The notebook uses code in `guiding_tool.py`, which depends on Python standard-library modules and the following packages:

- *NumPy* for numerical calculations, image processing, and Gaussian jitter fits
- *pandas* for telemetry tables, CSV input/output, and timestamp alignment
- *ASDF* and *roman_datamodels* to read Roman guide window products
- *Astropy* for timestamps and guide window image scaling
- *astroquery* to query MAST metadata and retrieve data products
- *Bokeh* for interactive plots and guide window displays
- *ipywidgets* and *IPython* for dashboard controls and notebook output
- *python-dotenv* to load MAST authentication settings from an environment file
- *PyERFA* for astronomical time support through Astropy
- *CRDS* to retrieve MA-table reference files for resultant timing
- *Pillow* (version 10.1 or later) and *imageio-ffmpeg* to create guide window movies; these are required only for movie export

## Instructions and Information

Execute the two code cells below to initialize the dashboard. Bokeh renders the plots and, if applicable, provides tools to pan, zoom, and inspect data.

### Dashboard Inputs

Enter a numeric visit ID in the “Visit ID” field—for example, `0100701001001002`. You can enter multiple comma-separated visit IDs; wildcards are not supported. Selecting a timing JSON file automatically fills the field with the visit IDs found in that file. You can edit the IDs before loading.

You can optionally provide a telemetry CSV, a visit timing JSON, and a directory containing guide window ASDF files. Each input bypasses its corresponding remote lookup: the timing JSON bypasses the MAST visit/exposure metadata query, the telemetry CSV bypasses engineering telemetry queries, and the guide window directory supplies local guide window products. For multi-visit telemetry replay, the CSV must include a `visit_id` column. A timing JSON used for detector-jitter filtering should include the per-exposure `guide_stars` mapping; older timing files without this metadata will not offer detector jitter sources.

You can enter absolute or relative paths, or use the “Browse” buttons to select files and directories on the notebook kernel’s filesystem. After browsing, click “Select” to put the chosen path in the text field.

### Metadata

After loading a visit, the dashboard displays information for the selected visit and exposure. The dashboard selects the first exposure by default. Displayed information includes:

- The selected visit ID and visit start time (UTC)
- The number of exposures in the visit
- The duration of the first SLEW mode block, from its first sample until telemetry reports a change out of SLEW. If the telemetry ends while still in SLEW, the duration is shown as a lower bound.
- The count of FACE estimate and variance values rejected during BAD quality periods, using `WFIF_FAC_ATT_SOL_QUALITY`
- The selected exposure ID, MA table name, and optical element
- The exposure duration
- The count of rejected FACE estimate and variance values during the selected exposure
- Initial delta, epsilon, and radial offsets in milliarcseconds (mas)

The initial offsets are calculated from the first nonzero FACE delta and epsilon values at or after the start of the first SLEW block. They are not necessarily measurements taken after the slew ends. The radial offset is the quadrature sum of the delta and epsilon offsets.

The Attitude Control System (ACS) uses delta (δ) and epsilon (ε) coordinates, measured in radians and converted to mas for the dashboard. The origin of the ε–δ plane is at the “WFI_CEN” position. These coordinates are approximately anti-parallel to the WFI science axes: +δ is similar to science −Y, and +ε is similar to science −X. For more information about the science coordinate system and the “WFI_CEN” position, see the RDox [Coordinate Systems](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products/coordinate-systems) article.

### NEA Tab

The “NEA” tab plots FACE delta and epsilon as a function of time. By default, “Whole Visit” is selected. Uncheck it to plot only the exposure selected in the “Exposure” dropdown.

A single-exposure plot includes 10 seconds before the exposure starts and 10 seconds after it ends. The whole-visit plot starts 20 seconds before the first SETTLE block, or at the visit start if there is no SETTLE block, and extends until 20 seconds after the last exposure.

Shaded regions indicate spacecraft mode: red for SLEW, orange for SETTLE, and green for inertial hold. Reaction-wheel zero-crossings, when present, are marked with dashed red lines. Hover over the FACE data to see NEA values and the number of stars the ACS used.

### Jitter Tab

This tab plots jitter from either FACE telemetry or detector centroid telemetry. You cannot select FACE and detector sources together, but you can select multiple detectors. Detector options are limited to detectors with a known, non-pseudo guide star ID for the selected exposure. A guide star ID beginning with `PGS` indicates a pseudo guide star and is excluded from detector jitter options.

Use the “Units” radio buttons to choose mas or pixels. FACE supports mas only. For detector centroids, the plotted coordinates are:

- X = `H_COORD + GW_H_CENTER`
- Y = `−(V_COORD + GW_V_CENTER)`

The full transformation from (H, V) to (X, Y) requires a shift in the origin; however, we plot the jitter centered on the mean, so the origin shift is unnecessary.

When mas is selected, the pixel values are multiplied by 110 mas/pixel. The axes are labeled X and Y for detector plots, and epsilon and delta for FACE plots.

The plot centers each source’s values on that source’s mean within the selected exposure. When you select multiple detectors, they appear in different colors. An ellipse is fit to each dataset; the legend reports its semi-major and semi-minor axes and position angle.

Use the “Highlight Resultant” dropdown to highlight an MA-table resultant. Its available jitter samples appear as larger cyan points.

### Guide Windows Tab

Guide window pixel data is not loaded by default. Click “Load guide windows” to display the selected exposure’s guide windows. If no local guide window directory is supplied, the products are retrieved from MAST; this can take several minutes, particularly for long visits.

The display arranges the detector windows in a grid matching the WFI focal plane. Guide window products marked as pseudo in their metadata are labeled “(pseudo).” A red cross shows the telemetry centroid when available; it is omitted for pseudo stars. Image values are scaled with ZScale, using one shared scale across the displayed windows.

Only tracking guide windows are shown. Guide window files also contain acquisition-window pixel data if you want to examine it separately. In imaging mode, tracking windows are 16 × 16 pixels; in spectroscopic mode, they are 128 rows × 32 columns (the “ACQ2” window), aligned with the dispersion direction. Spectroscopic windows also show the spectrum edge used for guiding.

To save the original guide window and FACE ASDF products for the selected exposure, enter an existing destination directory and click “Save GW and FACE ASDF files.”

To create a movie of the displayed guide windows, enter a filename and click “Make GW Movie.” You can set the frames per second (FPS); the default is 5 FPS. A progress bar appears while the movie is being created.

### Inertial Hold Report

The inertial hold report contains one row per exposure, including the IH start and end times and the interval from a known IH start to the exposure start (`ih_seconds`). Subsequent exposures in the same continuous IH interval show 0 seconds. A new IH interval has its own start time; if IH was already active at the first returned telemetry sample, its actual start time is unknown, and the delay is unavailable.

### Export Results Tab

Use this tab to save the telemetry table, inertial hold report, and visit/exposure information. Telemetry and IH reports are saved as CSV files; visit and exposure timing is saved as JSON. Timing exports include per-exposure guide star IDs obtained from MAST, allowing detector-jitter filtering during later replay. By default, existing files are not overwritten; select “Overwrite” to replace them.

### Recommended Diagnostics

1. Check whether an exposure contains FACE values rejected during BAD quality periods. A visit can contain rejected values outside its exposures without affecting the exposure data.
2. Review the initial delta and epsilon offsets. The dashboard calculates these from the first nonzero values at or after the first SLEW block begins; they are not necessarily post-slew settled-pointing measurements.
3. Reaction-wheel zero-crossings during an exposure may be evidence of degraded guiding stability.
4. Compare the jitter distribution for an MA table resultant with the rest of the exposure. For example, low scatter significantly offset from zero may indicate the observatory had not fully settled by that resultant.
5. Review the detector centroid overlays and guide window images. Large centroid changes relative to the star or spectral trace may indicate poor centroid quality. Also look for sources or detector artifacts that could bias centroid measurements.
6. A delay greater than 60 seconds between a known IH start and an exposure start may be indicative of guiding issues.

***

In [ ]:
from bokeh.io import output_notebook
import guiding_tool
output_notebook()

In [ ]:
dashboard = guiding_tool.create_dashboard()
display(dashboard)

## Help
For assistance with RIST, please contact the [Roman SOC Help Desk](https://stsci.service-now.com/roman).

## Additional Resources
- [WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products) article on RDox
- [Coordinate Systems](https://roman-docs.stsci.edu/data-handbook/wfi-data-levels-and-products/coordinate-systems) article on RDox

## About this Notebook
Disclaimer: This tool was developed using AI.

**Author:** Tyler Desjardins\
**Last Updated**: 2026 November

***

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>